### Xác minh chính sách theo địa bàn

Notebook này bổ sung lớp kiểm tra chính sách theo địa bàn cho dự án hộ nghèo/cận nghèo Đắk Nông.

**Nguyên tắc:**
- Giữ nguyên mapping chính sách theo nhóm hộ/cụm.
- Tách riêng phạm vi áp dụng chính sách theo địa bàn pháp lý.
- Model chỉ dùng để ưu tiên rà soát, không quyết định hộ hoặc địa bàn đủ điều kiện hưởng chính sách.
- Thêm lớp phù hợp theo đặc trưng địa bàn: văn bản chỉ được ưu tiên khi nhóm hộ/cụm nổi bật tại địa bàn có liên quan đến chính sách đó.
- Chỉ các dòng `TRANG_THAI_XAC_MINH = Đã xác minh` mới được đưa vào validation chính thức.

#### 1. Import thư viện và cấu hình


In [1]:
from pathlib import Path
import json
import shutil
import unicodedata

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = ROOT / "data" / "model"
POLICY_LOOKUP = MODEL_DIR / "province_policy_lookup.xlsx"
NOTEBOOK_PATH = ROOT / "notebooks" / "predictive_analysis.ipynb"
CLUSTERED_PATH = MODEL_DIR / "household_clustered.xlsx"
XGBOOST_RESULTS = MODEL_DIR / "xgboost_results.xlsx"

SCOPE_COLUMNS = [
    "SO_HIEU_VAN_BAN",
    "TEN_CHINH_SACH",
    "CAN_CU_DIEU_KHOAN_HO_TRO",
    "CAP_BAN_HANH",
    "THOI_GIAN_HIEU_LUC",
    "PHAM_VI_AP_DUNG",
    "HUYEN_AP_DUNG",
    "XA_AP_DUNG",
    "DOI_TUONG_AP_DUNG",
    "NHOM_HO_TRO_CHINH",
    "NGUON_XAC_MINH",
    "TRANG_THAI_XAC_MINH",
    "GHI_CHU_DIEU_KIEN",
    "LOAI_NGUON",
    "LINK_VAN_BAN",
    "TRANG_THAI_HIEU_LUC",
    "NGAY_XAC_MINH",
]

VALIDATED_STATUS = "Đã xác minh"
VALID_EFFECT_STATUS = "Còn hiệu lực"
ALL_AREA_VALUES = {"tat ca", "toan tinh", "toan huyen", "all", "*"}

# Căn cứ được quản lý theo số hiệu văn bản; chỉ dùng để đối chiếu, không tự động kết luận hộ đủ điều kiện.
POLICY_CITATION_BY_CODE = {
    "luatso51/2024/qh15": "Luật 51/2024/QH15: Điều 1 (sửa Điều 3, 12, 16-17, 21-23, 28 Luật BHYT).",
    "luatso39/2009/qh12": "Luật 39/2009/QH12: Điều 12-13 (chăm sóc sức khỏe), Điều 19 (chăm sóc tại cộng đồng).",
    "nghidinh20/2021/nd-cp": "Nghị định 20/2021/NĐ-CP: Điều 5 (đối tượng bảo trợ xã hội), Điều 6 (mức trợ cấp).",
    "nghidinh76/2024/nd-cp": "Nghị định 76/2024/NĐ-CP: Điều 2 khoản 1 (sửa Điều 4 Nghị định 20/2021/NĐ-CP).",
    "nghidinh238/2025/nd-cp": "Nghị định 238/2025/NĐ-CP: Điều 15 (miễn học phí), Điều 16-18 (giảm/hỗ trợ học phí), Điều 19-21 (hỗ trợ chi phí học tập).",
    "nghidinh66/2025/nd-cp": "Nghị định 66/2025/NĐ-CP: Điều 4 (điều kiện), Điều 6 (mức hưởng).",
    "thongtu23/2026/tt-bnnmt": "Thông tư 23/2026/TT-BNNMT: Điều 27 khoản 3 (hỗ trợ phát triển sản xuất giảm nghèo), Điều 28 (hỗ trợ tạo việc làm).",
    "quyetdinh16/2026/qd-ttg": "Quyết định 16/2026/QĐ-TTg: nguyên tắc, tiêu chí và định mức phân bổ ngân sách Chương trình mục tiêu quốc gia giai đoạn 2026-2030.",
}


#### 2. Hàm hỗ trợ


In [2]:
def safe_join(values, limit=4):
    seen = []
    for value in values.dropna().astype(str):
        if value and value not in seen:
            seen.append(value)
        if len(seen) >= limit:
            break
    return "; ".join(seen)


def normalize_place(value):
    text = "" if pd.isna(value) else str(value)
    text = text.replace("’", "'").replace("`", "'").strip().lower()
    for prefix in ["huyện ", "thành phố ", "thị xã ", "xã ", "phường ", "thị trấn "]:
        if text.startswith(prefix):
            text = text[len(prefix):]
    text = unicodedata.normalize("NFC", text)
    return " ".join(text.split())


def ascii_fold(value):
    text = "" if pd.isna(value) else str(value)
    text = unicodedata.normalize("NFD", text).lower()
    return "".join(c for c in text if unicodedata.category(c) != "Mn").strip()


def is_all_area(value):
    normalized = normalize_place(value)
    ascii_text = ascii_fold(normalized)
    return normalized in ALL_AREA_VALUES or ascii_text in ALL_AREA_VALUES


def citation_for_policy_code(value):
    key = ascii_fold(value).replace("đ", "d").replace(" ", "")
    return POLICY_CITATION_BY_CODE.get(key, "")


def split_policy_codes(values):
    codes = []
    for value in values.dropna().astype(str):
        for code in value.split(";"):
            code = code.strip()
            if code and code not in codes:
                codes.append(code)
    return codes


def write_or_replace_sheets(path, new_sheets):
    tmp_path = path.with_name(f"{path.stem}__tmp{path.suffix}")
    if path.exists() and path.stat().st_size > 0:
        shutil.copy2(path, tmp_path)
        with pd.ExcelWriter(tmp_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            for sheet_name, df in new_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    else:
        with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
            for sheet_name, df in new_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    tmp_path.replace(path)


#### 3. Load workbook policy


In [3]:
def rebuild_policy_workbook():
    nb = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    policy_cell = next(
        "".join(cell.get("source", []))
        for cell in nb["cells"]
        if "cluster_policy_rules = pd.DataFrame" in "".join(cell.get("source", []))
    )
    policy_cell = policy_cell.replace("display(cluster_policy_mapping)", "pass")
    policy_cell = policy_cell.replace("display(province_policy_lookup.head(10))", "pass")
    namespace = {
        "pd": pd,
        "ROOT": ROOT,
        "model_df": pd.read_excel(CLUSTERED_PATH, sheet_name="clustered"),
    }
    exec(policy_cell, namespace)
    return {
        "cluster_policy_mapping": namespace["cluster_policy_mapping"],
        "household_policy_linkage": namespace["household_policy_linkage"],
        "province_policy_lookup": namespace["province_policy_lookup"],
    }


def load_policy_workbook():
    try:
        if POLICY_LOOKUP.exists() and POLICY_LOOKUP.stat().st_size > 0:
            return pd.read_excel(POLICY_LOOKUP, sheet_name=None)
    except Exception:
        pass
    return rebuild_policy_workbook()


policy_workbook = load_policy_workbook()
policy = policy_workbook["province_policy_lookup"]
print("Loaded sheets:", list(policy_workbook.keys()))
print("province_policy_lookup rows:", len(policy))


Loaded sheets: ['cluster_policy_mapping', 'household_policy_linkage', 'province_policy_lookup', 'policy_geographic_scope', 'model_priority_area', 'policy_area_scope_validation', 'policy_area_feature_fit', 'policy_area_validation', 'policy_area_summary']
province_policy_lookup rows: 26953


#### 4. Chuẩn hóa `policy_geographic_scope`

`policy_geographic_scope` là bảng input research phạm vi áp dụng theo địa bàn. Các dòng chưa có nguồn xác minh vẫn được giữ lại, nhưng không đi vào validation chính thức.


In [4]:
def build_scope_template(policy):
    codes = split_policy_codes(policy["SO_HIEU_VAN_BAN_HO_TRO"])
    rows = []
    for code in codes:
        related = policy[
            policy["SO_HIEU_VAN_BAN_HO_TRO"].fillna("").astype(str).str.contains(code, regex=False)
        ]
        rows.append({
            "SO_HIEU_VAN_BAN": code,
            "TEN_CHINH_SACH": safe_join(related["TEN_VAN_BAN_HO_TRO"], limit=1),
            "CAN_CU_DIEU_KHOAN_HO_TRO": citation_for_policy_code(code),
            "CAP_BAN_HANH": "",
            "THOI_GIAN_HIEU_LUC": "",
            "PHAM_VI_AP_DUNG": "",
            "HUYEN_AP_DUNG": "",
            "XA_AP_DUNG": "",
            "DOI_TUONG_AP_DUNG": "",
            "NHOM_HO_TRO_CHINH": safe_join(related["NHOM_CHINH_SACH"], limit=3),
            "NGUON_XAC_MINH": "",
            "TRANG_THAI_XAC_MINH": "Chưa xác minh",
            "GHI_CHU_DIEU_KIEN": "Cần research phạm vi áp dụng theo địa bàn từ văn bản chính thức.",
            "LOAI_NGUON": "",
            "LINK_VAN_BAN": "",
            "TRANG_THAI_HIEU_LUC": "",
            "NGAY_XAC_MINH": "",
        })
    return pd.DataFrame(rows, columns=SCOPE_COLUMNS)


def load_scope(policy_workbook, policy):
    scope = policy_workbook.get("policy_geographic_scope")
    if scope is None or scope.empty:
        return build_scope_template(policy)
    for col in SCOPE_COLUMNS:
        if col not in scope.columns:
            scope[col] = ""
    return scope[SCOPE_COLUMNS].copy()


def normalize_scope(scope):
    normalized = scope.copy()
    for col in SCOPE_COLUMNS:
        normalized[col] = normalized[col].fillna("").astype(str).str.strip()
    normalized.loc[normalized["TRANG_THAI_XAC_MINH"] == "", "TRANG_THAI_XAC_MINH"] = "Chưa xác minh"
    normalized.loc[normalized["GHI_CHU_DIEU_KIEN"] == "", "GHI_CHU_DIEU_KIEN"] = "Cần research phạm vi áp dụng theo địa bàn từ văn bản chính thức."
    normalized["CAN_CU_DIEU_KHOAN_HO_TRO"] = normalized.apply(
        lambda row: row["CAN_CU_DIEU_KHOAN_HO_TRO"] or citation_for_policy_code(row["SO_HIEU_VAN_BAN"]),
        axis=1,
    )
    return normalized


def has_verified_scope(scope):
    required = ["SO_HIEU_VAN_BAN", "HUYEN_AP_DUNG", "XA_AP_DUNG", "NGUON_XAC_MINH"]
    if scope.empty:
        return pd.Series([], dtype=bool)
    status = scope["TRANG_THAI_XAC_MINH"].map(ascii_fold)
    effect_status = scope["TRANG_THAI_HIEU_LUC"].map(ascii_fold)
    return (
        scope[required].ne("").all(axis=1)
        & (status == ascii_fold(VALIDATED_STATUS))
        & ((effect_status == "") | (effect_status == ascii_fold(VALID_EFFECT_STATUS)))
    )


policy_scope = normalize_scope(load_scope(policy_workbook, policy))
print(policy_scope[["SO_HIEU_VAN_BAN", "TRANG_THAI_XAC_MINH", "LINK_VAN_BAN"]])
print("Verified scope rows:", has_verified_scope(policy_scope).sum())


             SO_HIEU_VAN_BAN TRANG_THAI_XAC_MINH LINK_VAN_BAN
0       Luật số 51/2024/QH15       Chưa xác minh             
1       Luật số 39/2009/QH12       Chưa xác minh             
2    Nghị định 20/2021/NĐ-CP       Chưa xác minh             
3    Nghị định 76/2024/NĐ-CP       Chưa xác minh             
4   Nghị định 238/2025/NĐ-CP       Chưa xác minh             
5    Nghị định 66/2025/NĐ-CP       Chưa xác minh             
6  Thông tư 23/2026/TT-BNNMT       Chưa xác minh             
7  Quyết định 16/2026/QĐ-TTg       Chưa xác minh             
Verified scope rows: 0


#### 5. Tạo `model_priority_area`

Bảng này chỉ cho biết địa bàn nào được dữ liệu/model ưu tiên rà soát. Nó không phải bằng chứng pháp lý về phạm vi áp dụng chính sách.


In [5]:
predictions = pd.read_excel(XGBOOST_RESULTS, sheet_name="poverty_predictions")
prediction_cols = ["HOUSEHOLD_ID", "prob_ngheo", "muc_do_rui_ro_ngheo", "muc_uu_tien_ra_soat"]
policy_with_predictions = policy.merge(
    predictions[[col for col in prediction_cols if col in predictions.columns]],
    on="HOUSEHOLD_ID",
    how="left",
)

overall_poor_rate = (policy_with_predictions["PHAN_LOAI_HO"] == "Hộ nghèo").mean()
overall_prob = policy_with_predictions["prob_ngheo"].mean()

geo = (
    policy_with_predictions.groupby(["HUYEN", "XA"], dropna=False)
    .agg(
        SO_HO_DIA_BAN=("HOUSEHOLD_ID", "nunique"),
        SO_HO_NGHEO_DIA_BAN=("PHAN_LOAI_HO", lambda s: (s == "Hộ nghèo").sum()),
        PROB_NGHEO_TB_DIA_BAN=("prob_ngheo", "mean"),
        SO_HO_RUI_RO_CAO_DIA_BAN=("prob_ngheo", lambda s: (s >= 0.7).sum()),
    )
    .reset_index()
)
geo["TY_LE_HO_NGHEO_DIA_BAN"] = (geo["SO_HO_NGHEO_DIA_BAN"] / geo["SO_HO_DIA_BAN"]).round(4)
geo["TY_LE_RUI_RO_CAO_DIA_BAN"] = (geo["SO_HO_RUI_RO_CAO_DIA_BAN"] / geo["SO_HO_DIA_BAN"]).round(4)

min_households = max(30, int(geo["SO_HO_DIA_BAN"].median()))
strong_mask = (
    (geo["SO_HO_DIA_BAN"] >= min_households)
    & (
        (geo["TY_LE_HO_NGHEO_DIA_BAN"] >= overall_poor_rate)
        | (geo["PROB_NGHEO_TB_DIA_BAN"] >= overall_prob)
        | (geo["SO_HO_RUI_RO_CAO_DIA_BAN"] > 0)
    )
)
geo["DIA_BAN_MO_HINH_UU_TIEN"] = strong_mask.map({True: "Có", False: "Không"})


def priority(row):
    if row["DIA_BAN_MO_HINH_UU_TIEN"] != "Có":
        return "Thấp"
    signals = sum([
        row["TY_LE_HO_NGHEO_DIA_BAN"] >= overall_poor_rate,
        row["PROB_NGHEO_TB_DIA_BAN"] >= overall_prob,
        row["SO_HO_RUI_RO_CAO_DIA_BAN"] > 0,
    ])
    return "Cao" if signals >= 2 else "Trung bình"


geo["MUC_UU_TIEN_MO_HINH"] = geo.apply(priority, axis=1)
model_priority_area = geo.sort_values(
    ["DIA_BAN_MO_HINH_UU_TIEN", "MUC_UU_TIEN_MO_HINH", "SO_HO_DIA_BAN"],
    ascending=[False, True, False],
)
print("model_priority_area rows:", len(model_priority_area))
print("priority areas:", (model_priority_area["DIA_BAN_MO_HINH_UU_TIEN"] == "Có").sum())


model_priority_area rows: 143
priority areas: 58


#### 6. Ghép phạm vi chính sách với địa bàn

Phần này hỗ trợ 3 trường hợp: toàn tỉnh, theo huyện, theo xã. Tên địa bàn được chuẩn hóa khi match, nhưng không sửa trực tiếp dữ liệu gốc.


In [6]:
def match_scope_to_geo(scope, geo):
    verified = scope[has_verified_scope(scope)].copy()
    output_cols = SCOPE_COLUMNS + list(geo.columns) + [
        "KET_QUA_KHOP_DIA_BAN",
        "LY_DO_KHONG_KHOP",
        "THUOC_PHAM_VI_CHINH_SACH",
        "LA_DIA_BAN_MO_HINH_UU_TIEN",
        "MUC_UU_TIEN_DOI_CHIEU",
    ]
    if verified.empty:
        return pd.DataFrame(columns=output_cols)

    rows = []
    geo_match = geo.copy()
    geo_match["_HUYEN_MATCH"] = geo_match["HUYEN"].map(normalize_place)
    geo_match["_XA_MATCH"] = geo_match["XA"].map(normalize_place)

    for _, rule in verified.iterrows():
        huyen_all = is_all_area(rule["HUYEN_AP_DUNG"])
        xa_all = is_all_area(rule["XA_AP_DUNG"])
        huyen = normalize_place(rule["HUYEN_AP_DUNG"])
        xa = normalize_place(rule["XA_AP_DUNG"])

        matched = geo_match.copy()
        if not huyen_all:
            matched = matched[matched["_HUYEN_MATCH"] == huyen]
        if not xa_all:
            matched = matched[matched["_XA_MATCH"] == xa]

        if matched.empty:
            rows.append({
                **rule.to_dict(),
                "KET_QUA_KHOP_DIA_BAN": "Không khớp",
                "LY_DO_KHONG_KHOP": "Không tìm thấy huyện/xã trong dữ liệu hộ sau chuẩn hóa tên địa bàn.",
                "THUOC_PHAM_VI_CHINH_SACH": "Có",
                "LA_DIA_BAN_MO_HINH_UU_TIEN": "",
                "MUC_UU_TIEN_DOI_CHIEU": "Cần kiểm tra tên địa bàn",
            })
            continue

        for _, area in matched.iterrows():
            area_dict = area.drop(labels=["_HUYEN_MATCH", "_XA_MATCH"]).to_dict()
            is_model_priority = area_dict["DIA_BAN_MO_HINH_UU_TIEN"] == "Có"
            if huyen_all and xa_all:
                match_result = "Toàn tỉnh"
                match_note = "Được mở rộng từ phạm vi toàn tỉnh."
            elif xa_all:
                match_result = "Theo huyện"
                match_note = "Được mở rộng cho toàn bộ xã có trong dữ liệu hộ của huyện áp dụng."
            else:
                match_result = "Khớp"
                match_note = ""
            rows.append({
                **rule.to_dict(),
                **area_dict,
                "KET_QUA_KHOP_DIA_BAN": match_result,
                "LY_DO_KHONG_KHOP": match_note,
                "THUOC_PHAM_VI_CHINH_SACH": "Có",
                "LA_DIA_BAN_MO_HINH_UU_TIEN": "Có" if is_model_priority else "Không",
                "MUC_UU_TIEN_DOI_CHIEU": "Cao" if is_model_priority else "Bình thường",
            })
    result = pd.DataFrame(rows)
    return result[[col for col in output_cols if col in result.columns]]


matched_scope = match_scope_to_geo(policy_scope, model_priority_area)
print("matched_scope rows:", len(matched_scope))
print(matched_scope["KET_QUA_KHOP_DIA_BAN"].value_counts(dropna=False) if len(matched_scope) else "No matched scope")


matched_scope rows: 0
No matched scope


#### 7. Tạo 4 bảng validation

- `policy_area_scope_validation`: grain `SO_HIEU_VAN_BAN x HUYEN x XA`, chỉ trả lời văn bản có thuộc phạm vi địa bàn không.
- `policy_area_feature_fit`: grain `SO_HIEU_VAN_BAN x HUYEN x XA x cụm/nhóm hộ`, trả lời văn bản có phù hợp đặc trưng hộ nổi bật tại địa bàn không.
- `policy_area_validation`: drill-through theo văn bản, địa bàn, nhóm hộ/cụm, đã có cờ ưu tiên đối chiếu.
- `policy_area_summary`: mỗi `HUYEN x XA` một dòng, tách chính sách đang áp dụng với chính sách nên ưu tiên theo đặc trưng.

In [7]:
def policy_code_key(value):
    return ascii_fold(value).replace("\u0111", "d")


def explode_policy_codes(df, source_col="SO_HIEU_VAN_BAN_HO_TRO"):
    exploded = df.copy()
    exploded["SO_HIEU_VAN_BAN"] = exploded[source_col].fillna("").astype(str).str.split(";")
    exploded = exploded.explode("SO_HIEU_VAN_BAN")
    exploded["SO_HIEU_VAN_BAN"] = exploded["SO_HIEU_VAN_BAN"].fillna("").astype(str).str.strip()
    exploded = exploded[exploded["SO_HIEU_VAN_BAN"] != ""]
    exploded["SO_HIEU_VAN_BAN_MATCH"] = exploded["SO_HIEU_VAN_BAN"].map(policy_code_key)
    return exploded


def fit_level(row):
    count = row.get("SO_HO_PHU_HOP_DAC_TRUNG", 0)
    ratio = row.get("TY_LE_HO_PHU_HOP_DAC_TRUNG", 0)
    if pd.isna(count) or count <= 0:
        return "Kh\u00f4ng n\u1ed5i b\u1eadt"
    if count >= 50 or ratio >= 0.25:
        return "Cao"
    if count >= 10 or ratio >= 0.10:
        return "Trung b\u00ecnh"
    return "Th\u1ea5p"


def review_priority(row):
    if row.get("DIA_BAN_CHINH_SACH_AP_DUNG") != "C\u00f3":
        return "Kh\u00f4ng"
    if row.get("MUC_PHU_HOP_DAC_TRUNG") in ["Cao", "Trung b\u00ecnh"]:
        return "C\u00f3"
    if row.get("MUC_PHU_HOP_DAC_TRUNG") == "Th\u1ea5p" and row.get("LA_DIA_BAN_MO_HINH_UU_TIEN") == "C\u00f3":
        return "C\u00f3"
    return "Kh\u00f4ng"


scope_key_cols = [
    *SCOPE_COLUMNS,
    "HUYEN",
    "XA",
    "SO_HO_DIA_BAN",
    "SO_HO_NGHEO_DIA_BAN",
    "TY_LE_HO_NGHEO_DIA_BAN",
    "PROB_NGHEO_TB_DIA_BAN",
    "DIA_BAN_MO_HINH_UU_TIEN",
    "MUC_UU_TIEN_MO_HINH",
    "KET_QUA_KHOP_DIA_BAN",
    "LY_DO_KHONG_KHOP",
    "THUOC_PHAM_VI_CHINH_SACH",
    "LA_DIA_BAN_MO_HINH_UU_TIEN",
    "MUC_UU_TIEN_DOI_CHIEU",
]

if matched_scope.empty:
    policy_area_feature_fit = pd.DataFrame()
    policy_area_validation = pd.DataFrame()
    policy_area_scope_validation = pd.DataFrame(columns=scope_key_cols)
    policy_area_summary = pd.DataFrame()
else:
    policy_area_scope_validation = (
        matched_scope[scope_key_cols]
        .drop_duplicates()
        .sort_values(["SO_HIEU_VAN_BAN", "HUYEN", "XA"])
    )

    matched_scope_for_join = matched_scope.copy()
    matched_scope_for_join["SO_HIEU_VAN_BAN_MATCH"] = matched_scope_for_join["SO_HIEU_VAN_BAN"].map(policy_code_key)

    household_policy_source = explode_policy_codes(policy_with_predictions)
    household_feature_cols = [
        "SO_HIEU_VAN_BAN_MATCH",
        "HUYEN",
        "XA",
        "CLUSTER_ID",
        "TEN_CUM",
        "NHOM_KHO_KHAN_CHINH",
        "NHOM_KHO_KHAN_PHU",
        "NHOM_CHINH_SACH",
        "YEU_TO_NOI_BAT",
    ]
    household_policy_fit = (
        household_policy_source.groupby(household_feature_cols, dropna=False)
        .agg(
            SO_HO_PHU_HOP_DAC_TRUNG=("HOUSEHOLD_ID", "nunique"),
            SO_HO_NGHEO_PHU_HOP=("PHAN_LOAI_HO", lambda s: (s == "H\u1ed9 ngh\u00e8o").sum()),
            PROB_NGHEO_TB_NHOM_PHU_HOP=("prob_ngheo", "mean"),
            SO_HO_RUI_RO_CAO_NHOM_PHU_HOP=("prob_ngheo", lambda s: (s >= 0.7).sum()),
        )
        .reset_index()
    )

    policy_area_feature_fit = matched_scope_for_join.merge(
        household_policy_fit,
        on=["SO_HIEU_VAN_BAN_MATCH", "HUYEN", "XA"],
        how="left",
    )
    policy_area_feature_fit["DIA_BAN_CHINH_SACH_AP_DUNG"] = policy_area_feature_fit["KET_QUA_KHOP_DIA_BAN"].map(
        lambda value: "C\u00f3" if value != "Kh\u00f4ng kh\u1edbp" else "Kh\u00f4ng"
    )
    policy_area_feature_fit["SO_HO_PHU_HOP_DAC_TRUNG"] = policy_area_feature_fit["SO_HO_PHU_HOP_DAC_TRUNG"].fillna(0).astype(int)
    policy_area_feature_fit["SO_HO_NGHEO_PHU_HOP"] = policy_area_feature_fit["SO_HO_NGHEO_PHU_HOP"].fillna(0).astype(int)
    policy_area_feature_fit["SO_HO_RUI_RO_CAO_NHOM_PHU_HOP"] = policy_area_feature_fit["SO_HO_RUI_RO_CAO_NHOM_PHU_HOP"].fillna(0).astype(int)
    policy_area_feature_fit["TY_LE_HO_PHU_HOP_DAC_TRUNG"] = (
        policy_area_feature_fit["SO_HO_PHU_HOP_DAC_TRUNG"] / policy_area_feature_fit["SO_HO_DIA_BAN"]
    ).fillna(0).round(4)
    policy_area_feature_fit["MUC_PHU_HOP_DAC_TRUNG"] = policy_area_feature_fit.apply(fit_level, axis=1)
    policy_area_feature_fit["CAN_DOI_CHIEU"] = policy_area_feature_fit.apply(review_priority, axis=1)
    policy_area_feature_fit["GHI_CHU_PHU_HOP_DAC_TRUNG"] = policy_area_feature_fit.apply(
        lambda row: (
            "\u01afu ti\u00ean r\u00e0 so\u00e1t v\u00ec v\u0103n b\u1ea3n thu\u1ed9c ph\u1ea1m vi \u0111\u1ecba b\u00e0n v\u00e0 c\u00f3 nh\u00f3m h\u1ed9/c\u1ee5m li\u00ean quan n\u1ed5i b\u1eadt t\u1ea1i \u0111\u1ecba b\u00e0n."
            if row["CAN_DOI_CHIEU"] == "C\u00f3"
            else "V\u0103n b\u1ea3n c\u00f3 th\u1ec3 thu\u1ed9c ph\u1ea1m vi \u0111\u1ecba b\u00e0n, nh\u01b0ng ch\u01b0a n\u1ed5i b\u1eadt theo \u0111\u1eb7c tr\u01b0ng nh\u00f3m h\u1ed9/c\u1ee5m trong d\u1eef li\u1ec7u."
        ),
        axis=1,
    )
    policy_area_feature_fit = policy_area_feature_fit.drop(columns=["SO_HIEU_VAN_BAN_MATCH"])

    policy_area_validation = policy_area_feature_fit.copy()
    policy_area_validation["GHI_CHU"] = (
        "Ph\u1ea1m vi \u00e1p d\u1ee5ng l\u1ea5y t\u1eeb policy_geographic_scope; m\u1ee9c \u01b0u ti\u00ean ch\u1ec9 t\u0103ng khi m\u00e3 v\u0103n b\u1ea3n kh\u1edbp v\u1edbi c\u1ee5m h\u1ed9 n\u1ed5i b\u1eadt t\u1ea1i \u0111\u1ecba b\u00e0n."
    )

    matched_validation = policy_area_validation[policy_area_validation["DIA_BAN_CHINH_SACH_AP_DUNG"] == "C\u00f3"].copy()
    prioritized_validation = matched_validation[matched_validation["CAN_DOI_CHIEU"] == "C\u00f3"].copy()

    legal_summary = (
        policy_area_scope_validation.groupby(["HUYEN", "XA"], dropna=False)
        .agg(
            CO_THUOC_DIA_BAN_CHINH_SACH=("THUOC_PHAM_VI_CHINH_SACH", "first"),
            CAC_CHINH_SACH_DANG_AP_DUNG=("TEN_CHINH_SACH", safe_join),
            SO_HO_NGHEO_DIA_BAN=("SO_HO_NGHEO_DIA_BAN", "first"),
            SO_HO_DIA_BAN=("SO_HO_DIA_BAN", "first"),
            MUC_UU_TIEN_MO_HINH=("MUC_UU_TIEN_MO_HINH", "first"),
            NGUON_XAC_MINH=("NGUON_XAC_MINH", safe_join),
        )
        .reset_index()
    )

    if prioritized_validation.empty:
        feature_summary = pd.DataFrame(columns=[
            "HUYEN",
            "XA",
            "CAC_CHINH_SACH_UU_TIEN_THEO_DAC_TRUNG",
            "NHOM_HO_TRO_UU_TIEN",
            "NHOM_HO_PHU_HOP",
            "DAC_TRUNG_HO_NOI_BAT",
            "SO_HO_PHU_HOP_DAC_TRUNG",
            "MUC_PHU_HOP_DAC_TRUNG",
        ])
    else:
        feature_summary = (
            prioritized_validation.groupby(["HUYEN", "XA"], dropna=False)
            .agg(
                CAC_CHINH_SACH_UU_TIEN_THEO_DAC_TRUNG=("TEN_CHINH_SACH", safe_join),
                NHOM_HO_TRO_UU_TIEN=("NHOM_HO_TRO_CHINH", safe_join),
                NHOM_HO_PHU_HOP=("TEN_CUM", safe_join),
                DAC_TRUNG_HO_NOI_BAT=("YEU_TO_NOI_BAT", safe_join),
                SO_HO_PHU_HOP_DAC_TRUNG=("SO_HO_PHU_HOP_DAC_TRUNG", "sum"),
                MUC_PHU_HOP_DAC_TRUNG=("MUC_PHU_HOP_DAC_TRUNG", "first"),
            )
            .reset_index()
        )

    policy_area_summary = legal_summary.merge(feature_summary, on=["HUYEN", "XA"], how="left")
    for col in [
        "CAC_CHINH_SACH_UU_TIEN_THEO_DAC_TRUNG",
        "NHOM_HO_TRO_UU_TIEN",
        "NHOM_HO_PHU_HOP",
        "DAC_TRUNG_HO_NOI_BAT",
        "MUC_PHU_HOP_DAC_TRUNG",
    ]:
        policy_area_summary[col] = policy_area_summary[col].fillna("Kh\u00f4ng n\u1ed5i b\u1eadt")
    policy_area_summary["SO_HO_PHU_HOP_DAC_TRUNG"] = policy_area_summary["SO_HO_PHU_HOP_DAC_TRUNG"].fillna(0).astype(int)
    policy_area_summary["MUC_UU_TIEN_RA_SOAT"] = policy_area_summary.apply(
        lambda row: "Cao" if row["MUC_UU_TIEN_MO_HINH"] == "Cao" and row["SO_HO_PHU_HOP_DAC_TRUNG"] > 0
        else "Trung b\u00ecnh" if row["SO_HO_PHU_HOP_DAC_TRUNG"] > 0
        else "B\u00ecnh th\u01b0\u1eddng",
        axis=1,
    )
    policy_area_summary = policy_area_summary.rename(columns={"SO_HO_DIA_BAN": "SO_HO_CAN_DOI_CHIEU"})
    priority_sort = {"Cao": 0, "Trung b?nh": 1, "B?nh th??ng": 2}
    policy_area_summary["_MUC_UU_TIEN_SORT"] = policy_area_summary["MUC_UU_TIEN_RA_SOAT"].map(priority_sort).fillna(3)
    policy_area_summary = (
        policy_area_summary.sort_values(
            ["_MUC_UU_TIEN_SORT", "SO_HO_PHU_HOP_DAC_TRUNG", "SO_HO_CAN_DOI_CHIEU"],
            ascending=[True, False, False],
        )
        .drop(columns=["_MUC_UU_TIEN_SORT"])
    )

print("policy_area_scope_validation rows:", len(policy_area_scope_validation))
print("policy_area_feature_fit rows:", len(policy_area_feature_fit))
print("policy_area_validation rows:", len(policy_area_validation))
print("policy_area_summary rows:", len(policy_area_summary))

policy_area_scope_validation rows: 0
policy_area_feature_fit rows: 0
policy_area_validation rows: 0
policy_area_summary rows: 0


#### 8. Kiểm tra chất lượng


In [8]:
if not policy_area_scope_validation.empty:
    key_cols = ["SO_HIEU_VAN_BAN", "HUYEN", "XA"]
    print("scope unique keys:", policy_area_scope_validation[key_cols].drop_duplicates().shape)
    print(policy_area_scope_validation.groupby(key_cols, dropna=False).size().value_counts().sort_index())

if not policy_area_feature_fit.empty:
    fit_key_cols = ["SO_HIEU_VAN_BAN", "HUYEN", "XA", "CLUSTER_ID"]
    fit_keys = policy_area_feature_fit[fit_key_cols].drop_duplicates()
    prioritized = policy_area_feature_fit[policy_area_feature_fit["CAN_DOI_CHIEU"] == "C\u00f3"]
    print("feature-fit unique keys:", fit_keys.shape)
    print("feature-fit rows:", policy_area_feature_fit.shape)
    print("prioritized feature-fit rows:", len(prioritized))
    print("prioritized policies:", prioritized["SO_HIEU_VAN_BAN"].nunique())

if not policy_area_validation.empty:
    key_cols = ["SO_HIEU_VAN_BAN", "HUYEN", "XA"]
    print("detail unique policy-area keys:", policy_area_validation[key_cols].drop_duplicates().shape)
    print("detail rows:", policy_area_validation.shape)

print("summary unique areas:", policy_area_summary[["HUYEN", "XA"]].drop_duplicates().shape if not policy_area_summary.empty else (0, 2))
print("scope link missing:", policy_scope["LINK_VAN_BAN"].fillna("").astype(str).eq("").sum())
print("verified scope rows:", has_verified_scope(policy_scope).sum())

summary unique areas: (0, 2)
scope link missing: 8
verified scope rows: 0


#### 9. Ghi output


In [9]:
new_policy_sheets = {
    "policy_geographic_scope": policy_scope,
    "model_priority_area": model_priority_area,
    "policy_area_scope_validation": policy_area_scope_validation,
    "policy_area_feature_fit": policy_area_feature_fit,
    "policy_area_validation": policy_area_validation,
    "policy_area_summary": policy_area_summary,
}
write_or_replace_sheets(POLICY_LOOKUP, new_policy_sheets)
print(f"Updated validation sheets: {POLICY_LOOKUP}")

Updated validation sheets: c:\NguyenHoangThanhTam\data\model\province_policy_lookup.xlsx
